## Step 2: Data Cleaning

In [1]:
import pandas as pd
df = pd.read_csv("../data/Loan.csv")

### Health Check

In [2]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Experience > Age:", (df["Experience"] > df["Age"]).sum())

# Any negative numbers? (empty result = good)
neg = df.select_dtypes("number").lt(0).sum()
print(neg[neg > 0])

Missing values: 0
Duplicate rows: 0
Experience > Age: 0
Series([], dtype: int64)


### Leakage check

#### Correlation and approval check

In [3]:
suspects = ["InterestRate", "BaseInterestRate", "MonthlyLoanPayment", "TotalDebtToIncomeRatio"]
fair = ["CreditScore", "DebtToIncomeRatio", "PaymentHistory"]   # for comparison

print(df[suspects + fair].corrwith(df["RiskScore"]).sort_values())
print(df.groupby("LoanApproved")["RiskScore"].mean())

CreditScore              -0.240198
PaymentHistory           -0.016992
MonthlyLoanPayment        0.115501
BaseInterestRate          0.256233
InterestRate              0.268203
DebtToIncomeRatio         0.326500
TotalDebtToIncomeRatio    0.342643
dtype: float64
LoanApproved
0    54.106307
1    40.133389
Name: RiskScore, dtype: float64


#### Test the suspect debt ratio

In [4]:
gap = df["TotalDebtToIncomeRatio"] - df["DebtToIncomeRatio"]
payment_share = df["MonthlyLoanPayment"] / df["MonthlyIncome"]
print(gap.corr(payment_share))      # close to 1 = it includes the payment

0.8606972289171317


#### Tiebreaker: what do the leaky columns buy?

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

leaky = ["LoanApproved", "InterestRate", "BaseInterestRate",
         "MonthlyLoanPayment", "TotalDebtToIncomeRatio"]
X_all = df.select_dtypes("number").drop(columns="RiskScore")

for name, X in {"with suspects": X_all, "without": X_all.drop(columns=leaky)}.items():
    r2 = cross_val_score(LinearRegression(), X, df["RiskScore"], cv=5, scoring="r2").mean()
    print(name, round(r2, 3))

with suspects 0.813
without 0.642


#### Repeated information

In [6]:
# NetWorth = Assets - Liabilities ?
nw_diff = (df["TotalAssets"] - df["TotalLiabilities"] - df["NetWorth"]).abs()
print("NetWorth rows that don't match:", (nw_diff > 1).sum())

# MonthlyIncome = AnnualIncome / 12 ?
mi_diff = (df["AnnualIncome"] / 12 - df["MonthlyIncome"]).abs()
print("MonthlyIncome rows that don't match:", (mi_diff > 1).sum())

NetWorth rows that don't match: 4801
MonthlyIncome rows that don't match: 195


In [7]:
calc_nw = df["TotalAssets"] - df["TotalLiabilities"]
bad = nw_diff > 1

print("Corr with Assets - Liabilities:", round(df["NetWorth"].corr(calc_nw), 4))
print("Negative: NetWorth =", (df["NetWorth"] < 0).sum(), "| Assets - Liabilities =", (calc_nw < 0).sum())
print(df.loc[bad, ["TotalAssets", "TotalLiabilities", "NetWorth"]].assign(calc=calc_nw[bad]).head(8))

# MonthlyIncome: are the 195 rows the capped ones?
bad_mi = mi_diff > 1
print(df.loc[bad_mi, ["AnnualIncome", "MonthlyIncome"]].head(5))
print("Max MonthlyIncome:", df["MonthlyIncome"].max())

Corr with Assets - Liabilities: 0.959
Negative: NetWorth = 0 | Assets - Liabilities = 4629
    TotalAssets  TotalLiabilities  NetWorth    calc
2         25176            128874      5205 -103698
8         29632             49940      7711  -20308
12        16204             35493      5231  -19289
16        11162             89196      7021  -78034
22        23984             27250      2482   -3266
23        14710             53225      6903  -38515
31        56887             90622      3068  -33735
46        17748            117394      4958  -99646
     AnnualIncome  MonthlyIncome
227         29512    1250.000000
287        179664    8772.416667
337         64483    3053.166667
349         86286    4153.000000
508         72965    3276.166667
Max MonthlyIncome: 25000.0


#### Drop the columns

In [8]:
drop_reasons = {
    "LoanApproved": "Leakage: decision made from risk",
    "InterestRate": "Leakage: set from risk",
    "BaseInterestRate": "Leakage: set from risk",
    "MonthlyLoanPayment": "Leakage: built from the interest rate",
    "TotalDebtToIncomeRatio": "Leakage: includes loan payment (gap test = 0.86)",
    "NetWorth": "Repeats TotalAssets - TotalLiabilities",
    "MonthlyIncome": "Repeats AnnualIncome / 12",
    "ApplicationDate": "A date, not a driver of risk",
}

print("Before:", df.shape)
df = df.drop(columns=list(drop_reasons))     # errors if a name is wrong. Fix the name.
print("After: ", df.shape)

Before: (20000, 36)
After:  (20000, 28)


#### Outliers

In [9]:
tail_cols = ["AnnualIncome", "LoanAmount", "TotalAssets", "TotalLiabilities"]
summary = pd.DataFrame({
    "median": df[tail_cols].median(),
    "p99.9": df[tail_cols].quantile(0.999),
    "max": df[tail_cols].max(),
})
summary["max / p99.9"] = summary["max"] / summary["p99.9"]
print(summary.round(2))

print("Rows above p99.9 income:", (df["AnnualIncome"] > df["AnnualIncome"].quantile(0.999)).sum())

                   median       p99.9      max  max / p99.9
AnnualIncome      48566.0   300000.00   485341         1.62
LoanAmount        21914.5   110702.73   184732         1.67
TotalAssets       60699.0  1280720.17  2619627         2.05
TotalLiabilities  22203.0   486366.49  1417302         2.91
Rows above p99.9 income: 7


Kept all outliers. Max values are under 3x the 99.9th percentile in every checked column, and only 7 income rows exceed it. Assets and liabilities are right-skewed but smooth. Will recheck residuals in Step 4 and cap or log-transform if a few rows distort the fit.

#### Text to numbers

In [10]:
df_labeled = df.copy()    # keep the text version for EDA charts

cat_cols = df.select_dtypes(exclude="number").columns.tolist()
print(cat_cols)    # expect: employment, education, marital, home ownership, loan purpose

df_clean = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)
print(df_clean.shape)

['EmploymentStatus', 'EducationLevel', 'MaritalStatus', 'HomeOwnershipStatus', 'LoanPurpose']
(20000, 39)


In [11]:
assert df_clean.select_dtypes(exclude="number").shape[1] == 0   # all numeric
assert df_clean.isna().sum().sum() == 0                          # no missing

print(df_clean.filter(like="EmploymentStatus").head(3))          # see the new 0/1 columns

   EmploymentStatus_Self-Employed  EmploymentStatus_Unemployed
0                               0                            0
1                               0                            0
2                               0                            0


#### Save and log

In [12]:
df_labeled.to_csv("../data/loan_clean_labeled.csv", index=False)   # before encoding
df_clean.to_csv("../data/loan_clean.csv", index=False)             # after encoding
print(df_labeled.shape, df_clean.shape)

(20000, 28) (20000, 39)


In [13]:
print("df_labeled:", df_labeled.shape)
print(df_labeled.columns.tolist())

print("\ndf_clean:", df_clean.shape)
print(df_clean.columns.tolist())

# What changed between the two?
removed = [c for c in df_labeled.columns if c not in df_clean.columns]
added   = [c for c in df_clean.columns if c not in df_labeled.columns]
print("\nRemoved (text columns):", removed)
print("Added (0/1 columns):", added)

df_labeled: (20000, 28)
['Age', 'AnnualIncome', 'CreditScore', 'EmploymentStatus', 'EducationLevel', 'Experience', 'LoanAmount', 'LoanDuration', 'MaritalStatus', 'NumberOfDependents', 'HomeOwnershipStatus', 'MonthlyDebtPayments', 'CreditCardUtilizationRate', 'NumberOfOpenCreditLines', 'NumberOfCreditInquiries', 'DebtToIncomeRatio', 'BankruptcyHistory', 'LoanPurpose', 'PreviousLoanDefaults', 'PaymentHistory', 'LengthOfCreditHistory', 'SavingsAccountBalance', 'CheckingAccountBalance', 'TotalAssets', 'TotalLiabilities', 'UtilityBillsPaymentHistory', 'JobTenure', 'RiskScore']

df_clean: (20000, 39)
['Age', 'AnnualIncome', 'CreditScore', 'Experience', 'LoanAmount', 'LoanDuration', 'NumberOfDependents', 'MonthlyDebtPayments', 'CreditCardUtilizationRate', 'NumberOfOpenCreditLines', 'NumberOfCreditInquiries', 'DebtToIncomeRatio', 'BankruptcyHistory', 'PreviousLoanDefaults', 'PaymentHistory', 'LengthOfCreditHistory', 'SavingsAccountBalance', 'CheckingAccountBalance', 'TotalAssets', 'TotalLiabil

# Step 2 Final Log: Data Cleaning

## Cleaning summary

**Input:** `Loan.csv` (20,000 rows, 36 columns)
**Output:** `loan_clean_labeled.csv` (20,000 x 28, text labels) and `loan_clean.csv` (20,000 x 39, all numeric)

### Health check
No missing values, no duplicate rows, no negative values, no rows where `Experience` > `Age`.

### Dropped columns (8)

| Dropped column | Reason |
|:---|:---|
| `LoanApproved` | Leakage: decision made from risk (approved avg risk 40.1 vs rejected 54.1) |
| `InterestRate`, `BaseInterestRate` | Leakage: set after risk is judged |
| `MonthlyLoanPayment` | Leakage: built from the interest rate |
| `TotalDebtToIncomeRatio` | Leakage: includes the loan payment (gap test corr = 0.86) |
| `NetWorth` | Repeats `TotalAssets` - `TotalLiabilities` (corr 0.96). Negatives replaced by small positives |
| `MonthlyIncome` | Repeats `AnnualIncome` / 12 (99% of rows match) |
| `ApplicationDate` | A date, not a driver of risk |

**Leakage evidence:** 5-fold CV R² was 0.813 with the leaky columns and 0.642 without. The 0.642 is the honest baseline (numeric columns only, before encoding).

### Outliers
Kept all. Max values are under 3x the 99.9th percentile in every checked column, and only 7 income rows exceed it. Assets and liabilities are right-skewed but smooth. Will recheck residuals in Step 4 and cap or log-transform if a few rows distort the fit.

### Encoding
One-hot encoding with `drop_first=True` and `dtype=int` on 5 columns: `EmploymentStatus`, `EducationLevel`, `MaritalStatus`, `HomeOwnershipStatus`, `LoanPurpose`. This added 16 columns. Baseline for `EmploymentStatus` is Employed.

### Handoff
- `03_eda.ipynb` loads `../data/loan_clean_labeled.csv`
- `04_modeling.ipynb` loads `../data/loan_clean.csv` (38 features + `RiskScore`)
